In [1]:
# The purpose of this notebook is to try experimenting with aplha (noise level per data point)
# aplha should be the variance (sd^2)

In [2]:
# import the data

from pathlib import Path
import pandas as pd
import numpy as np

# Load the compiled data
project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
path = project_root / 'data' / 'clean' / 'train_scaled.csv'

df = pd.read_csv(path)

formula_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]

rng = np.random.RandomState(1)
X = df[formula_cols].fillna(0).to_numpy()
y = df['Wet Strength Mean'].to_numpy()
  
print(df.head(10))

   TEM   Joncryl     NC513  PEG-DE  NGDE  Avg_Density_g_cm3  Avg_Density SE  \
0  0.5       NaN       NaN     NaN   NaN           0.192000        0.002667   
1  0.5 -0.959381       NaN     NaN   NaN           0.195000        0.007263   
2  0.5 -0.520342       NaN     NaN   NaN           0.159000        0.005033   
3  0.5  0.211389       NaN     NaN   NaN           0.159933        0.004410   
4  0.5  0.943121       NaN     NaN   NaN           0.201000        0.008817   
5  0.5  2.406584       NaN     NaN   NaN           0.202000        0.006807   
6 -2.0       NaN       NaN     NaN   NaN           0.298930        0.001613   
7 -2.0       NaN -1.778900     NaN   NaN           0.360035        0.001748   
8 -2.0       NaN -1.452995     NaN   NaN           0.355557        0.002226   
9 -2.0       NaN -0.366643     NaN   NaN           0.358055        0.005455   

   Dry Strength Mean  Dry Strength SE  Wet Strength Mean  Wet Strength SE  \
0          53.674733         1.008133          23.680

In [3]:
# get alpha

# import the data

from pathlib import Path
import pandas as pd
import numpy as np

# Load the compiled data
project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
path = project_root / 'data' / 'clean' / 'Dry_strength_Replicates_scaled.csv'

df = pd.read_csv(path)
print("Data shape:", df.shape)
print("\nFirst few rows of the data:")
print(df.head())

Data shape: (25, 4)

First few rows of the data:
   Dry Strength 1  Dry Strength 2  Dry Strength 3  Variance
0       -0.654719       -0.556178       -0.611894  0.002441
1       -0.171142       -0.367608       -0.437274  0.019046
2       -0.593961       -0.651029       -0.751818  0.006389
3       -0.613716       -0.544410       -0.606066  0.001444
4        0.112459       -0.310057        0.140225  0.063674


In [4]:
# get the variance in the alpha variable

alpha = df['Variance'].to_numpy()

In [5]:
# break X and y into LOOCV folds, refit the model on each fold/ print
# evaluation metrics

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import r2_score

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

# LOOCV cross-validation
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()

val_predictions = []
val_actuals = []
train_r2_scores = []

fold_num = 1
for train_idx, val_idx in loo.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    alpha_train = alpha[train_idx]
    
    # Fit the model using noise variances for the training observations only
    model = GaussianProcessRegressor(
        kernel=RBF(length_scale=1.0, length_scale_bounds=(1e-10, 1e4)),
        normalize_y=True,
        n_restarts_optimizer=10,
        alpha=alpha_train,
        random_state=0,
    )
    model.fit(X_train, y_train)
    
    # Store validation prediction
    val_pred = model.predict(X_val)
    val_predictions.append(val_pred[0])
    val_actuals.append(y_val[0])
    
    # Evaluate on training fold
    train_pred = model.predict(X_train)
    train_r2 = r2_score(y_train, train_pred)
    train_r2_scores.append(train_r2)
    
    print(f"Fold {fold_num}: Val Prediction = {val_pred[0]:.3f}, Actual = {y_val[0]:.3f}")
    fold_num += 1

# Calculate R² across ALL out-of-fold predictions
val_r2 = r2_score(val_actuals, val_predictions)

# Calculate averages
avg_train_r2 = np.mean(train_r2_scores)

# Create summary table
results_df = pd.DataFrame({
    'Error Source': ['Goal Performance', 'Train Error', 'Validation Error (LOOCV)', 'Test Error'],
    'Value': [
        'R² = .8',
        f'R² = {avg_train_r2:.3f}',
        f'R² = {val_r2:.3f}',
        ''
    ]
})

print("\n" + "="*50)
print("EVALUATION SUMMARY")
print("="*50)
print(results_df.to_string(index=False))

Fold 1: Val Prediction = 21.807, Actual = 23.680
Fold 2: Val Prediction = 16.850, Actual = 24.258
Fold 3: Val Prediction = 26.393, Actual = 21.329
Fold 4: Val Prediction = 25.697, Actual = 20.184
Fold 5: Val Prediction = 14.738, Actual = 27.585
Fold 6: Val Prediction = 35.094, Actual = 27.922
Fold 7: Val Prediction = 66.066, Actual = 51.134
Fold 8: Val Prediction = 63.410, Actual = 82.812
Fold 9: Val Prediction = 85.168, Actual = 77.134
Fold 10: Val Prediction = 53.192, Actual = 76.406
Fold 11: Val Prediction = 22.714, Actual = 35.350
Fold 12: Val Prediction = 26.068, Actual = 19.560
Fold 13: Val Prediction = 23.558, Actual = 19.610
Fold 14: Val Prediction = 23.001, Actual = 16.730
Fold 15: Val Prediction = 23.316, Actual = 16.970
Fold 16: Val Prediction = 19.667, Actual = 17.040
Fold 17: Val Prediction = 18.292, Actual = 19.620
Fold 18: Val Prediction = 29.347, Actual = 29.008
Fold 19: Val Prediction = 24.293, Actual = 27.360
Fold 20: Val Prediction = 17.970, Actual = 20.682
Fold 21: 